## Importing the necessary Libraries for the Model 

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import classification_report, confusion_matrix

from PIL import Image

## Now let's Define the Dataset path for training and testing of the data 

In [2]:
train_dataset=keras.utils.image_dataset_from_directory(
    directory="../data/raw/training_set",
    labels="inferred",
    label_mode="int",
    batch_size=32,
    image_size=(256, 256),
    shuffle=True,
    seed=42)
test_dataset=keras.utils.image_dataset_from_directory(
    directory="../data/raw/test_set",
    labels="inferred",
    label_mode="int",
    batch_size=32,
    image_size=(256, 256),
    shuffle=True,
    seed=42)
print(train_dataset.class_names)

Found 8005 files belonging to 2 classes.
Found 2023 files belonging to 2 classes.
['cats', 'dogs']


## Let's visualize some random sample images

In [ ]:
plt.figure(figsize=(10, 10))

for images, labels in train_dataset.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)

        plt.imshow(images[i].numpy().astype("uint8"))

        plt.title(train_dataset.class_names[labels[i]])

        plt.axis("off")

plt.show()


### Let's check the images shape and labels . 

In [ ]:
for images, labels in train_dataset.take(1):
    print("Images batch shape:", images.shape)
    print("Labels batch shape:", labels.shape)

    print("\nImage data type:", images.dtype)
    print("Labels:", labels[:10].numpy())

#### Here (32, 256, 256, 3) the tuple elements have their own values. 
#### Batch size(32), The model processes 32 images at a time before updating its learning weights.
#### RGB channels (3): Each image has 3 color channels — Red, Green, and Blue — used to represent a colored image.
#### Width (256): Each image is resized to 256 pixels wide.
#### Height (256): Each image is resized to 256 pixels high.

In [7]:
for images, labels in train_dataset.take(1):
    print("Minimum pixel value:", images.numpy().min())
    print("Maximum pixel value:", images.numpy().max())

Minimum pixel value: 0.0
Maximum pixel value: 255.0


### The Output means the images currently use the standard RGB pixel range . 0(black)---------255(white)

##### Since the values are in range of 0-256 , we need to normalize them to a scale of 0-1 . We carry out these steps beacuse normalization makes pixel values smaller and more consistent, helping the neural network train more efficiently.

In [ ]:
normalization_layer = layers.Rescaling(1./255)

## Apply the normalization layer to the training and testing datasets 
train_dataset = train_dataset.map(
    lambda images, labels: (normalization_layer(images), labels)
)

test_dataset = test_dataset.map(
    lambda images, labels: (normalization_layer(images), labels)
)

# Now let's check the pixel values after normalization
for images, labels in train_dataset.take(1):
    print("Minimum pixel value after normalization:", images.numpy().min())
    print("Maximum pixel value after normalization:", images.numpy().max())

### Now let's optimize the dataset pipeline using cache() and prefetch().
### cache() → Keeps processed data available for faster reuse.
### shuffle(1000) → Randomly mixes training images.
### prefetch() → Prepares the next batch while the model is training on the current batch.

In [9]:
AUTOTUNE = tf.data.AUTOTUNE

# Apply caching, shuffling, and prefetching to the training dataset for performance optimization
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)

test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

## Now we have the pipeline as:
#### Raw Images
####    ↓
#### Resize → 256 × 256
####     ↓
#### Batch → 32 images
####     ↓
#### Normalize → 0 to 1
####     ↓
#### Cache + Shuffle + Prefetch
####     ↓
#### Ready for CNN 

### Data Augmentation helps the model generalize instead of memorizing the training images. so, We'll apply augmentation only to the training images, not the test dataset.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Let's visualize the effect of data augmentation on a sample image from the training dataset
plt.figure(figsize=(10, 10))

for images, labels in train_dataset.take(1):
    
    for i in range(9):
        augmented_image = data_augmentation(
            tf.expand_dims(images[i], 0)
        )
        
        plt.subplot(3, 3, i + 1)
        plt.imshow(augmented_image[0])
        plt.axis("off")

plt.show()

### What each augmentation does.
#### RandomFlip → randomly flips images horizontally.
#### RandomRotation → slightly rotates images.
#### RandomZoom → randomly zooms in/out.

### Important:-
#### We don't permanently save the augmented images. The augmentation happens dynamically during training, and the generated versions are processed in different batches. When the model has processed the entire training dataset once, that complete cycle is called an epoch. The augmented images are temporary and exist only while being processed; the epoch itself does not store them.


# Let's Initialize the model

In [ ]:
model = keras.Sequential([
    
    # Data Augmentation
    data_augmentation,
    
    # First Convolution Block
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    
    # Second Convolution Block
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    
    # Third Convolution Block
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    
    # Convert feature maps into a single vector
    layers.Flatten(),
    
    # Fully Connected Layer
    layers.Dense(128, activation="relu"),
    
    # Reduce Overfitting
    layers.Dropout(0.5),
    
    # Output Layer
    layers.Dense(1, activation="sigmoid")
])

# check The architecture of the model
model.summary()

### CNN Model Architecture

The model is built using `keras.Sequential()`, which means the input passes through each layer sequentially.

- **`data_augmentation`**: Applies random transformations such as flipping, rotation, and zooming to improve model generalization and reduce overfitting.

- **`Conv2D(32, (3, 3), activation="relu")`**:
  - `32` → Number of filters used to detect different features.
  - `(3, 3)` → Size of the convolution kernel that scans the image.
  - `relu` → Activation function that introduces non-linearity.

- **`MaxPooling2D((2, 2))`**:
  - `(2, 2)` → Pooling window size.
  - Reduces the spatial dimensions of feature maps while retaining important features.

- **`Conv2D(64, (3, 3), activation="relu")`**:
  - `64` → Increases the number of filters so the network can learn more feature patterns.
  - `(3, 3)` → Uses a 3×3 kernel.
  - `relu` → Adds non-linearity.

- **`MaxPooling2D((2, 2))`**: Further reduces the spatial dimensions of the feature maps.

- **`Conv2D(128, (3, 3), activation="relu")`**:
  - `128` → Allows the network to learn more complex features.
  - `(3, 3)` → Uses a 3×3 kernel.
  - `relu` → Adds non-linearity.

- **`MaxPooling2D((2, 2))`**: Reduces the spatial dimensions again while preserving important features.

- **`Flatten()`**: Converts the final multi-dimensional feature maps into a single one-dimensional vector so they can be passed to the fully connected layer.

- **`Dense(128, activation="relu")`**:
  - `128` → Number of neurons in the fully connected layer.
  - `relu` → Activation function used to learn complex relationships between the extracted features.

- **`Dropout(0.5)`**:
  - `0.5` → Randomly disables approximately 50% of the neurons during training.
  - Helps reduce overfitting.

- **`Dense(1, activation="sigmoid")`**:
  - `1` → One output neuron because this is a binary classification problem (cat or dog).
  - `sigmoid` → Produces an output between 0 and 1, representing the prediction for the two classes.

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

## Now Train the model

In [ ]:
dog_cat_model = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=10
)

Epoch 1/10


c:\Projects_Tracking\Dog_Cat_Classification\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


251/251 ━━━━━━━━━━━━━━━━━━━━ 201s 794ms/step - accuracy: 0.5369 - loss: 0.7102 - val_accuracy: 0.5685 - val_loss: 0.6867
Epoch 2/10
 22/251 ━━━━━━━━━━━━━━━━━━━━ 2:54 762ms/step - accuracy: 0.5421 - loss: 0.6889